In [1]:
import pandas as pd
import numpy as np

# LOAD TRAINING DATA
df = pd.read_csv("train.csv", parse_dates=["datetime"])

# Parsing Date and Time
df["hour"] = df["datetime"].dt.hour
df["day"] = df["datetime"].dt.day
df["month"] = df["datetime"].dt.month
df["year"] = df["datetime"].dt.year
df = df.drop(columns=["datetime"])


# Remove columns that cause leakage because in real life we wont have these fields as input, because if
# we do have them, that means we have the answer itself ,  count = casual + registered
df = df.drop(columns=["casual", "registered"])

# print(df.head())


In [2]:
# WHY ONE HOT ENCODE??
# One-hot encoding prevents linear regression from assuming an incorrect numeric ordering between categories.
# If we use weather = 1, 2, 3, 4 directly
# Is weather = 4 really "4 times worse/better" than weather = 1?

def one_hot_encode(df, col_name):
    unique_vals = sorted(df[col_name].unique())
    for val in unique_vals:
        df[f"{col_name}_{val}"] = (df[col_name] == val).astype(int)
    df.drop(columns=[col_name], inplace=True)
    return df

categorical_cols = ["season", "holiday", "workingday", "weather",
                    "month", "hour", "day", "year"]

for col in categorical_cols:
    df = one_hot_encode(df, col)

# print(df.head())
print(df.shape)


(10886, 74)


In [3]:
# 80–20 split

X = df.drop(columns=["count"]).values.astype(float)
y = df["count"].values.reshape(-1, 1)

def manual_split(X, y, test_ratio=0.2):
    n = len(X)
    idx = np.arange(n)
    np.random.shuffle(idx)

    split_point = int(n * (1 - test_ratio))
    train_idx = idx[:split_point]
    test_idx = idx[split_point:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X_train, X_test, y_train, y_test = manual_split(X, y, test_ratio=0.2)

print("Shapes:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


Shapes:
X_train: (8708, 73)
X_test : (2178, 73)
y_train: (8708, 1)
y_test : (2178, 1)


In [4]:
# Why standardize
# If features are on very different scales, X transpose X becomes ill-conditioned.
# Polynomials explode without scaling If humidity = 80:
# humidity⁴ = 40,960,000

def standardize(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-8   # avoid division by zero

    X_train_std = (X_train - mean) / std
    X_test_std  = (X_test - mean) / std   # IMPORTANT: use TRAIN MEAN + STD

    return X_train_std, X_test_std, mean, std

X_train_std, X_test_std, mean_vec, std_vec = standardize(X_train, X_test)

print("Standardized shapes:")
print(X_train_std.shape, X_test_std.shape)

Standardized shapes:
(8708, 73) (2178, 73)


In [5]:
def add_bias(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

# Add bias column
X_train_b = add_bias(X_train_std)
X_test_b  = add_bias(X_test_std)

print("X_train with bias:", X_train_b.shape)
print("X_test with bias :", X_test_b.shape)


X_train with bias: (8708, 74)
X_test with bias : (2178, 74)


In [6]:
# HELPER FUNCTIONS

# Solve for beta
def normal_equation(X, y):
    return np.linalg.pinv(X.T @ X) @ (X.T @ y)    #Moore–Penrose pseudoinverse

# Prediction
def predict(X, beta):
    return X @ beta

# Mean Squared Error
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

# R² Score
def r2(y_true, y_pred):
    ssr = np.sum((y_true - y_pred)**2)
    sst = np.sum((y_true - np.mean(y_true))**2)
    return 1 - ssr/sst


In [7]:

# Train
beta_linear = normal_equation(X_train_b, y_train)

# Predict on test set
y_pred_linear = predict(X_test_b, beta_linear)

# Metrics
mse_linear = mse(y_test, y_pred_linear)
r2_linear = r2(y_test, y_pred_linear)

print("Baseline Linear Regression Results:")
print("MSE:", mse_linear)
print("R² :", r2_linear)

Baseline Linear Regression Results:
MSE: 10107.501278867616
R² : 0.6887840853963982


In [8]:
def poly_no_interactions(X, degree):
    feats = [X]  # original features
    for p in range(2, degree + 1):
        feats.append(X ** p)   # element-wise exponent
    return np.hstack(feats)

def train_poly_model(degree, X_train_std, X_test_std):
    # Expand features
    Xtr_poly = poly_no_interactions(X_train_std, degree)
    Xte_poly = poly_no_interactions(X_test_std, degree)

    # Add bias
    Xtr_b = add_bias(Xtr_poly)
    Xte_b = add_bias(Xte_poly)

    # Train
    beta = normal_equation(Xtr_b, y_train)

    # Predict
    y_pred = predict(Xte_b, beta)

    # Metrics
    M = mse(y_test, y_pred)
    R = r2(y_test, y_pred)

    print(f"Polynomial degree {degree} (no interactions):")
    print("MSE:", M)
    print("R² :", R)
    print("----------------------------------------------")

    return M, R


In [9]:
print("\n=== Polynomial Regression WITHOUT Interaction Terms ===\n")

results_poly = {}

for d in [2, 3, 4]:
    results_poly[d] = train_poly_model(d, X_train_std, X_test_std)



=== Polynomial Regression WITHOUT Interaction Terms ===

Polynomial degree 2 (no interactions):
MSE: 10039.627774675613
R² : 0.6908739505472075
----------------------------------------------
Polynomial degree 3 (no interactions):
MSE: 9920.799924386924
R² : 0.6945327300108612
----------------------------------------------
Polynomial degree 4 (no interactions):
MSE: 9914.688681370706
R² : 0.6947208987809839
----------------------------------------------


In [10]:
def quadratic_with_interactions(X):
    X = X.copy()
    N, d = X.shape

    # Linear + squared terms
    features = [X, X**2]

    # Interaction terms
    inter_terms = []
    for i in range(d):
        for j in range(i+1, d):
            inter_terms.append((X[:, i] * X[:, j]).reshape(-1, 1))

    # Combine all
    if inter_terms:
        inter_concat = np.hstack(inter_terms)
        return np.hstack([X, X**2, inter_concat])
    else:
        return np.hstack([X, X**2])


In [11]:
def train_quadratic_interaction_model(X_train_std, X_test_std):
    # Expand
    Xtr_q = quadratic_with_interactions(X_train_std)
    Xte_q = quadratic_with_interactions(X_test_std)

    # Add bias
    Xtr_b = add_bias(Xtr_q)
    Xte_b = add_bias(Xte_q)

    # Train using normal equation
    beta = normal_equation(Xtr_b, y_train)

    # Predict
    y_pred = predict(Xte_b, beta)

    # Metrics
    M = mse(y_test, y_pred)
    R = r2(y_test, y_pred)

    print("\nQuadratic Model WITH Interaction Terms:")
    print("MSE:", M)
    print("R² :", R)
    print("==============================================")

    return M, R


In [12]:
M_quad, R_quad = train_quadratic_interaction_model(X_train_std, X_test_std)




Quadratic Model WITH Interaction Terms:
MSE: 2846.974840740687
R² : 0.9123399686560494


In [13]:
results = {}

# Baseline Linear
results["Linear Regression"] = (mse_linear, r2_linear)

# Polynomial no-interactions
for d in [2, 3, 4]:
    M, R = results_poly[d]
    results[f"Poly degree {d} (no interactions)"] = (M, R)

# Quadratic with interactions
results["Quadratic (with interactions)"] = (M_quad, R_quad)

print(f"{'Model':40s} {'MSE':15s} {'R²'}")
print("-" * 70)

for model_name, (M, R) in results.items():
    print(f"{model_name:40s} {M:<15.4f} {R:.4f}")


Model                                    MSE             R²
----------------------------------------------------------------------
Linear Regression                        10107.5013      0.6888
Poly degree 2 (no interactions)          10039.6278      0.6909
Poly degree 3 (no interactions)          9920.7999       0.6945
Poly degree 4 (no interactions)          9914.6887       0.6947
Quadratic (with interactions)            2846.9748       0.9123
